# Fase 2: Feature Engineering — Structural & Bilingual

**Notebook ke-2** dari pipeline pelatihan model deteksi tulisan AI.

| Checkpoint | File | Keterangan |
|---|---|---|
| Input | `data_cleaned.parquet` | Output dari Notebook 1 (Preprocessing) |
| Output | `features_structural.parquet` | 29 fitur numerik per teks |
| Config | `feature_config.json` | Konfigurasi fitur aktif & urutan kolom kanonik |

**Variabel yang dihasilkan notebook ini:**
- `df_data` — DataFrame long-format (dimuat dari checkpoint)
- `wordlists` — Dict semua wordlist bilingual yang sudah diproses ke set/list
- `feature_df` — DataFrame 29 fitur hasil ekstraksi

**Catatan Desain:**
- Semua 29 fitur disimpan sebagai kolom numerik `float64`.
- Fitur `contraction_count` bersifat EN-only: teks dengan `language == 'id'` mendapat nilai `0.0`.
- Proses ekstraksi dilakukan per-chunk (default 10.000 baris) untuk ketahanan terhadap crash.
- Pilihan fitur aktif disimpan di `feature_config.json` dan wajib digunakan oleh Notebook 4 (Training).


In [13]:
import sys
import subprocess

required = ["pandas", "numpy", "ipywidgets", "pyarrow", "fastparquet", "matplotlib", "seaborn"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Memasang {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import pandas as pd
import numpy as np
import json
import re
import os
import gc
import time
from datetime import datetime
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
import seaborn as sns

print(f"Library berhasil diimpor.")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")


Library berhasil diimpor.
Pandas  : 2.3.3
NumPy   : 2.3.5


## 1. Load Checkpoint dari Fase 1

Memuat `data_cleaned.parquet` yang dihasilkan oleh Notebook 1.
File ini harus sudah tersedia. Jika belum, jalankan `01_preprocessing.ipynb` terlebih dahulu.


In [14]:
CHECKPOINT_INPUT  = "data_cleaned.parquet"
WORDLIST_DIR      = "wordlists"
CHUNKS_DIR        = "feat_chunks"
CHECKPOINT_OUTPUT = "features_structural.parquet"
FEATURE_CONFIG_FILE = "feature_config.json"

assert os.path.exists(CHECKPOINT_INPUT), (
    f"File tidak ditemukan: {CHECKPOINT_INPUT}\n"
    "Jalankan 01_preprocessing.ipynb dan simpan checkpoint-nya terlebih dahulu."
)

print(f"Memuat {CHECKPOINT_INPUT}...")
t0 = time.time()
df_data = pd.read_parquet(CHECKPOINT_INPUT)
df_data = df_data.reset_index(drop=True)
elapsed = time.time() - t0

print(f"Selesai dalam {elapsed:.2f} detik.")
print(f"Jumlah baris   : {len(df_data):,}")
print(f"Kolom tersedia : {list(df_data.columns)}")
print()
print("Distribusi source:")
display(df_data["source"].value_counts().rename("Jumlah").to_frame())
print("Distribusi language:")
display(df_data["language"].value_counts().rename("Jumlah").to_frame())
display(df_data.head(3))


Memuat data_cleaned.parquet...
Selesai dalam 0.90 detik.
Jumlah baris   : 100,000
Kolom tersedia : ['id', 'text_clean', 'source', 'instructions', 'language']

Distribusi source:


,Jumlah
source,
ai,50000
human,50000


Distribusi language:


,Jumlah
language,
en,100000


,id,text_clean,source,instructions,language
0,317fc428-de31-4bec-a35e-41688ec60018,"Having looked into family records, I believe I...",ai,Task: \n\n- Research LOCATION_NAME and the att...,en
1,7bedae35-f300-40e7-8b90-1d2647a45f19,"Similarly, in relationships, mistakes are inev...",ai,"Task: \n\n1. Analyze the quote ""A problem is a...",en
2,9f2a29ab-97d4-414d-ab74-f9a0d941f575,"You also get more noticed, and people will lik...",human,Task: Research the benefits of being an action...,en


## 2. Load Wordlist Bilingual

Memuat dan memproses 10 file JSON dari folder `wordlists/` menjadi struktur yang dioptimalkan
untuk pencarian cepat (Python `set` untuk kata/frasa, `list` untuk pasangan dan regex).

Folder wordlist: `wordlists/` (hardcoded — tidak perlu konfigurasi).


In [15]:
def _flat_set(data, key):
    """Ambil list dari dict[key], kembalikan sebagai set lowercase."""
    raw = data.get(key, [])
    if isinstance(raw, list):
        return set(w.lower() for w in raw)
    return set()

def _nested_set(data, key):
    """Flatten dict-of-lists atau list-of-lists menjadi satu set lowercase."""
    result = set()
    nested = data.get(key, {})
    if isinstance(nested, dict):
        for cat_words in nested.values():
            if isinstance(cat_words, list):
                result.update(w.lower() for w in cat_words)
    elif isinstance(nested, list):
        result.update(w.lower() for w in nested)
    return result

def load_wordlists(wordlist_dir="wordlists"):
    """Load semua 10 wordlists dan preproses menjadi set/list untuk pencarian cepat."""
    wl = {}
    p = Path(wordlist_dir)

    # 1. ai_overrepresented_vocab.json flat list en[] + id[]
    with open(p / "ai_overrepresented_vocab.json", encoding="utf-8") as f:
        d = json.load(f)
    wl["ai_vocab_en"] = _flat_set(d, "en")
    wl["ai_vocab_id"] = _flat_set(d, "id")

    # 2. hedging_phrases.json flat list en[] + id[]
    with open(p / "hedging_phrases.json", encoding="utf-8") as f:
        d = json.load(f)
    wl["hedging_en"] = _flat_set(d, "en")
    wl["hedging_id"] = _flat_set(d, "id")

    # 3. transition_phrases.json nested dict per kategori (addition, contrast, ...)
    with open(p / "transition_phrases.json", encoding="utf-8") as f:
        d = json.load(f)
    wl["transition_en"] = _nested_set(d, "en")
    wl["transition_id"] = _nested_set(d, "id")

    # 4. formulaic_phrases.json nested dict per kategori (openers, conclusions, ...)
    with open(p / "formulaic_phrases.json", encoding="utf-8") as f:
        d = json.load(f)
    wl["formulaic_en"] = _nested_set(d, "en")
    wl["formulaic_id"] = _nested_set(d, "id")

    # 5. certainty_openers.json flat list en[] + id[]
    with open(p / "certainty_openers.json", encoding="utf-8") as f:
        d = json.load(f)
    wl["certainty_en"] = _flat_set(d, "en")
    wl["certainty_id"] = _flat_set(d, "id")

    # 6. vague_attribution.json flat list en[] + id[]
    with open(p / "vague_attribution.json", encoding="utf-8") as f:
        d = json.load(f)
    wl["vague_en"] = _flat_set(d, "en")
    wl["vague_id"] = _flat_set(d, "id")

    # 7. formal_register_pairs.json {"en": [{formal,informal},...], "id": [...], ...}
    with open(p / "formal_register_pairs.json", encoding="utf-8") as f:
        d = json.load(f)
    wl["formal_pairs_en"] = d.get("en", [])
    wl["formal_pairs_id"] = d.get("id", [])

    # 8. negative_parallelism.json flat list en[] + id[]
    with open(p / "negative_parallelism.json", encoding="utf-8") as f:
        d = json.load(f)
    wl["neg_parallel_en"] = _flat_set(d, "en")
    wl["neg_parallel_id"] = _flat_set(d, "id")

    # 9. copypaste_artifacts.json {"patterns": [{name, regex, severity, ...}]}
    with open(p / "copypaste_artifacts.json", encoding="utf-8") as f:
        d = json.load(f)
    raw_patterns = d.get("patterns", d) if isinstance(d, dict) else d
    # Precompile regex untuk performa
    compiled_patterns = []
    for pat in raw_patterns:
        try:
            compiled_patterns.append({
                "name": pat.get("name", ""),
                "severity": pat.get("severity", "moderate"),
                "_compiled": re.compile(pat["regex"], re.IGNORECASE | re.MULTILINE)
            })
        except (re.error, KeyError):
            pass  # Pattern invalid lewati
    wl["artifact_patterns"] = compiled_patterns

    # 10. contractions.json {"en": {"common": [...], "negation": [...], "informal": [...]}}
    with open(p / "contractions.json", encoding="utf-8") as f:
        d = json.load(f)
    contractions_flat = set()
    en_block = d.get("en", d)
    if isinstance(en_block, dict):
        for cat_words in en_block.values():
            if isinstance(cat_words, list):
                contractions_flat.update(w.lower() for w in cat_words)
    elif isinstance(en_block, list):
        contractions_flat.update(w.lower() for w in en_block)
    wl["contractions_flat"] = contractions_flat

    return wl

# Eksekusi
wordlists = load_wordlists(WORDLIST_DIR)
print("Wordlists berhasil dimuat:")
print(f"  ai_vocab       : {len(wordlists['ai_vocab_en'])} EN / {len(wordlists['ai_vocab_id'])} ID entri")
print(f"  hedging        : {len(wordlists['hedging_en'])} EN / {len(wordlists['hedging_id'])} ID entri")
print(f"  transition     : {len(wordlists['transition_en'])} EN / {len(wordlists['transition_id'])} ID entri")
print(f"  formulaic      : {len(wordlists['formulaic_en'])} EN / {len(wordlists['formulaic_id'])} ID entri")
print(f"  certainty      : {len(wordlists['certainty_en'])} EN / {len(wordlists['certainty_id'])} ID entri")
print(f"  vague_attrib   : {len(wordlists['vague_en'])} EN / {len(wordlists['vague_id'])} ID entri")
print(f"  formal_pairs   : {len(wordlists['formal_pairs_en'])} EN / {len(wordlists['formal_pairs_id'])} ID pasangan")
print(f"  neg_parallel   : {len(wordlists['neg_parallel_en'])} EN / {len(wordlists['neg_parallel_id'])} ID entri")
print(f"  artifact_regex : {len(wordlists['artifact_patterns'])} patterns (precompiled)")
print(f"  contractions   : {len(wordlists['contractions_flat'])} entri (EN only)")


Wordlists berhasil dimuat:
  ai_vocab       : 158 EN / 82 ID entri
  hedging        : 71 EN / 54 ID entri
  transition     : 115 EN / 111 ID entri
  formulaic      : 134 EN / 117 ID entri
  certainty      : 66 EN / 60 ID entri
  vague_attrib   : 88 EN / 75 ID entri
  formal_pairs   : 95 EN / 46 ID pasangan
  neg_parallel   : 38 EN / 42 ID entri
  artifact_regex : 25 patterns (precompiled)
  contractions   : 105 entri (EN only)


## 3. Konfigurasi Fitur Aktif

Pilih fitur mana yang akan diekstrak. Konfigurasi ini disimpan ke `feature_config.json`
dan **wajib dimuat oleh Notebook 4 (Training)** untuk menjaga konsistensi urutan kolom.

Urutan dalam `ALL_FEATURES` adalah urutan kanonik — jangan diubah antar notebook.

Tips:
- Aktifkan semua fitur untuk hasil terbaik.
- Nonaktifkan fitur tertentu untuk eksperimen ablasi (membandingkan kontribusi setiap fitur).


In [16]:
# Daftar kanonik semua 29 fitur URUTAN INI HARUS KONSISTEN ANTAR NOTEBOOK
ALL_FEATURES = [
    # Grup A: Language-Agnostic Structural (20 fitur)
    "sentence_count",           # 1  Jumlah kalimat
    "avg_sentence_length",      # 2  Rata-rata panjang kalimat (kata)
    "sentence_length_std",      # 3  Std dev panjang kalimat (rendah = AI)
    "paragraph_count",          # 4  Jumlah paragraf (split double newline)
    "paragraph_length_std",     # 5  Std dev panjang paragraf (rendah = AI)
    "unique_word_ratio",        # 6  TTR: unique_words / total_words
    "em_dash_density",          # 7  Jumlah em-dash (—) per 1000 karakter
    "colon_density",            # 8  Jumlah titik dua (:) per 1000 karakter
    "semicolon_density",        # 9  Jumlah titik koma (;) per 1000 karakter
    "exclamation_density",      # 10 Jumlah tanda seru (!) per 1000 karakter
    "question_density",         # 11 Jumlah tanda tanya (?) per 1000 karakter
    "bullet_list_count",        # 12 Jumlah bullet / numbered list items
    "bold_pattern_count",       # 13 Jumlah pola **bold** atau __bold__
    "header_pattern_count",     # 14 Jumlah header markdown (# ## dll)
    "avg_word_length",          # 15 Panjang kata rata-rata (karakter)
    "comma_density",            # 16 Jumlah koma (,) per 1000 karakter
    "parenthesis_density",      # 17 Jumlah tanda kurung () per 1000 karakter
    "contraction_count",        # 18 Jumlah kontraksi EN only, 0.0 untuk ID
    "number_density",           # 19 Jumlah angka per 1000 karakter
    "capital_ratio",            # 20 Rasio huruf kapital dari total huruf

    # Grup B: Bilingual Vocabulary (6 fitur dari wordlists)
    "ai_vocab_density",         # 21 Densitas kata AI-overrepresented per total kata
    "hedging_density",          # 22 Densitas frasa hedging per total kata
    "transition_density",       # 23 Densitas transisi per total kata
    "formulaic_score",          # 24 Densitas frasa formulaik per total kata
    "certainty_opener",         # 25 Count frasa pembuka antusias (di 200 char pertama)
    "vague_attribution_density",# 26 Densitas atribusi samar per total kata

    # Grup C: Pattern-Based (3 fitur numerik biasa)
    "not_only_but_also",        # 27 Count pola parallelisme negatif
    "formal_tone_score",        # 28 Rasio formal / (formal + informal)
    "copypaste_artifact_count", # 29 Count regex artefak copy-paste AI
]

FEATURE_GROUPS = {
    "Grup A Structural (Language-Agnostic, 20 fitur)": ALL_FEATURES[:20],
    "Grup B Bilingual Vocabulary (6 fitur)":           ALL_FEATURES[20:26],
    "Grup C Pattern-Based (3 fitur)":                  ALL_FEATURES[26:],
}

# Load konfigurasi yang tersimpan (jika ada)
_active_defaults = ALL_FEATURES.copy()
if os.path.exists(FEATURE_CONFIG_FILE):
    try:
        with open(FEATURE_CONFIG_FILE, encoding="utf-8") as f:
            _saved_cfg = json.load(f)
        _active_defaults = _saved_cfg.get("active_features", ALL_FEATURES)
        print(f"Konfigurasi dimuat dari {FEATURE_CONFIG_FILE}: "
              f"{len(_active_defaults)}/{len(ALL_FEATURES)} fitur aktif.")
    except Exception as e:
        print(f"Gagal membaca konfigurasi lama ({e}). Menggunakan default (semua fitur aktif).")
else:
    print("Tidak ada feature_config.json sebelumnya. Semua fitur aktif secara default.")

# Buat widget checkbox per grup
_checkboxes = {}
_group_vboxes = []

for grp_name, feat_list in FEATURE_GROUPS.items():
    _cbs = []
    for feat in feat_list:
        cb = widgets.Checkbox(
            value=(feat in _active_defaults),
            description=feat,
            indent=False,
            layout=widgets.Layout(width="330px")
        )
        _checkboxes[feat] = cb
        _cbs.append(cb)
    _group_vboxes.append(widgets.VBox(
        [widgets.HTML(f"<b style='color:#4a90e2;font-size:13px'>{grp_name}</b>")] + _cbs,
        layout=widgets.Layout(border="1px solid #3a3a5a", padding="8px", margin="4px 4px")
    ))

btn_save_cfg    = widgets.Button(description="Simpan Konfigurasi", button_style="success", icon="save",
                                  layout=widgets.Layout(width="200px"))
btn_select_all  = widgets.Button(description="Pilih Semua", button_style="info",    icon="check-square",
                                  layout=widgets.Layout(width="140px"))
btn_deselect_all = widgets.Button(description="Hapus Semua", button_style="warning", icon="square",
                                   layout=widgets.Layout(width="140px"))
out_cfg = widgets.Output()

def _save_config(b):
    active = [f for f in ALL_FEATURES if _checkboxes[f].value]
    config = {
        "active_features": active,
        "feature_count": len(active),
        "all_features_canonical": ALL_FEATURES,
        "saved_at": datetime.now().isoformat(),
        "notes": "Generated by 02_feature_engineering.ipynb. Wajib konsisten dengan Notebook 4."
    }
    with open(FEATURE_CONFIG_FILE, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)
    with out_cfg:
        clear_output()
        print(f"feature_config.json disimpan: {len(active)}/{len(ALL_FEATURES)} fitur aktif.")

def _select_all(b):
    for cb in _checkboxes.values(): cb.value = True

def _deselect_all(b):
    for cb in _checkboxes.values(): cb.value = False

btn_save_cfg.on_click(_save_config)
btn_select_all.on_click(_select_all)
btn_deselect_all.on_click(_deselect_all)

display(
    widgets.HBox(_group_vboxes, layout=widgets.Layout(flex_flow="row wrap")),
    widgets.HBox([btn_select_all, btn_deselect_all, btn_save_cfg]),
    out_cfg
)


Konfigurasi dimuat dari feature_config.json: 23/29 fitur aktif.


Output()

## 4. Fungsi Ekstraksi Fitur

Mendefinisikan semua fungsi yang digunakan untuk mengekstrak 29 fitur dari setiap teks.
Sel ini harus dijalankan sebelum melakukan ekstraksi batch.


In [17]:
def count_phrases(text_lower, phrase_set):
    """
    Hitung jumlah kemunculan phrase/kata dari phrase_set di dalam text_lower.
    - Multi-word phrase: substring search biasa.
    - Single word: word-boundary regex untuk akurasi.
    """
    count = 0
    for phrase in phrase_set:
        phrase = phrase.lower()
        if " " in phrase:
            count += text_lower.count(phrase)
        else:
            count += len(re.findall(r"\b" + re.escape(phrase) + r"\b", text_lower))
    return count


def compute_formal_score(text_lower, formal_pairs):
    """
    Hitung rasio penggunaan kata formal vs informal dari list pasangan kata.
    Mengembalikan formal_count / (formal_count + informal_count), atau 0.0 jika tidak ada.
    """
    formal_total = 0
    informal_total = 0
    for pair in formal_pairs:
        f = pair.get("formal", "").lower()
        i = pair.get("informal", "").lower()
        if f:
            if " " in f:
                formal_total += text_lower.count(f)
            else:
                formal_total += len(re.findall(r"\b" + re.escape(f) + r"\b", text_lower))
        if i:
            if " " in i:
                informal_total += text_lower.count(i)
            else:
                informal_total += len(re.findall(r"\b" + re.escape(i) + r"\b", text_lower))
    total = formal_total + informal_total
    return formal_total / total if total > 0 else 0.0


In [18]:
# Regex patterns dikompile sekali di luar fungsi untuk performa
_RE_SENTENCES   = re.compile(r"[.!?]+")
_RE_WORDS       = re.compile(r"\b\w+\b")
_RE_BULLET      = re.compile(r"^\s*([\u2022\-\*]|\d+\.?)\s+", re.MULTILINE)
_RE_BOLD        = re.compile(r"\*\*[^*]+\*\*|__[^_]+__")
_RE_HEADER      = re.compile(r"^#{1,6}\s+", re.MULTILINE)
_RE_NUMBER      = re.compile(r"\b\d+\b")
_EM_DASH        = "\u2014"


def extract_features(text, lang="en", wl=None):
    """
    Ekstrak semua 29 fitur dari satu teks.

    Args:
        text (str) : Teks yang sudah dibersihkan (dari kolom text_clean).
        lang (str) : Bahasa teks "en" atau "id". Default "en".
        wl   (dict): Wordlists yang sudah di-load dan diproses.

    Returns:
        dict: {feature_name: float_value} untuk semua 29 fitur di ALL_FEATURES.
    """
    # Teks kosong atau bukan string
    if not isinstance(text, str) or not text.strip():
        return {feat: 0.0 for feat in ALL_FEATURES}

    text_lower  = text.lower()
    char_count  = max(len(text), 1)

    # Tokenisasi
    sentences   = [s.strip() for s in _RE_SENTENCES.split(text) if s.strip()]
    words       = _RE_WORDS.findall(text_lower)
    word_count  = max(len(words), 1)
    paragraphs  = [p.strip() for p in re.split(r"\n\n+", text) if p.strip()]

    features = {}

    # ─GRUP A: Structural Features (20) ────────────────────────────────────

    features["sentence_count"]      = float(len(sentences))

    sent_lens = [len(_RE_WORDS.findall(s)) for s in sentences]
    features["avg_sentence_length"] = float(np.mean(sent_lens)) if sent_lens else 0.0
    features["sentence_length_std"] = float(np.std(sent_lens))  if len(sent_lens) > 1 else 0.0

    features["paragraph_count"]      = float(len(paragraphs))
    para_lens = [len(_RE_WORDS.findall(p)) for p in paragraphs]
    features["paragraph_length_std"] = float(np.std(para_lens)) if len(para_lens) > 1 else 0.0

    features["unique_word_ratio"]    = len(set(words)) / word_count

    features["em_dash_density"]      = text.count(_EM_DASH) / char_count * 1000
    features["colon_density"]        = text.count(":") / char_count * 1000
    features["semicolon_density"]    = text.count(";") / char_count * 1000
    features["exclamation_density"]  = text.count("!") / char_count * 1000
    features["question_density"]     = text.count("?") / char_count * 1000

    features["bullet_list_count"]    = float(len(_RE_BULLET.findall(text)))
    features["bold_pattern_count"]   = float(len(_RE_BOLD.findall(text)))
    features["header_pattern_count"] = float(len(_RE_HEADER.findall(text)))

    features["avg_word_length"]      = float(np.mean([len(w) for w in words])) if words else 0.0
    features["comma_density"]        = text.count(",") / char_count * 1000
    features["parenthesis_density"]  = (text.count("(") + text.count(")")) / char_count * 1000

    # Kontraksi: EN-only. Teks ID mendapat 0.0 sesuai keputusan desain.
    if lang == "en" and wl:
        features["contraction_count"] = float(
            sum(1 for w in words if w in wl["contractions_flat"])
        )
    else:
        features["contraction_count"] = 0.0

    features["number_density"]       = len(_RE_NUMBER.findall(text)) / char_count * 1000

    alpha_chars = [c for c in text if c.isalpha()]
    features["capital_ratio"] = (
        sum(1 for c in alpha_chars if c.isupper()) / max(len(alpha_chars), 1)
    )

    # ─GRUP B: Bilingual Vocabulary Features (6) ───────────────────────────

    lang_key = lang if lang in ("en", "id") else "en"

    if wl:
        ai_vocab = wl[f"ai_vocab_{lang_key}"]
        features["ai_vocab_density"] = sum(1 for w in words if w in ai_vocab) / word_count

        features["hedging_density"] = (
            count_phrases(text_lower, wl[f"hedging_{lang_key}"]) / word_count
        )
        features["transition_density"] = (
            count_phrases(text_lower, wl[f"transition_{lang_key}"]) / word_count
        )
        features["formulaic_score"] = (
            count_phrases(text_lower, wl[f"formulaic_{lang_key}"]) / word_count
        )

        # certainty_opener: hanya cek 200 karakter pertama teks
        text_start = text_lower[:200]
        features["certainty_opener"] = float(
            count_phrases(text_start, wl[f"certainty_{lang_key}"])
        )

        features["vague_attribution_density"] = (
            count_phrases(text_lower, wl[f"vague_{lang_key}"]) / word_count
        )

        # ─GRUP C: Pattern-Based Features (3) ──────────────────────────────

        features["not_only_but_also"] = float(
            count_phrases(text_lower, wl[f"neg_parallel_{lang_key}"])
        )

        features["formal_tone_score"] = compute_formal_score(
            text_lower, wl[f"formal_pairs_{lang_key}"]
        )

        artifact_count = 0
        for pat in wl["artifact_patterns"]:
            compiled = pat.get("_compiled")
            if compiled:
                artifact_count += len(compiled.findall(text))
        features["copypaste_artifact_count"] = float(artifact_count)

    else:
        # Wordlists tidak tersedia: set semua vocab/pattern features ke 0
        for feat in ALL_FEATURES[20:]:
            features[feat] = 0.0

    return features


# Quick sanity check pada 1 sampel
_sample_row  = df_data.iloc[0]
_sample_feat = extract_features(
    _sample_row.get("text_clean", ""),
    lang=_sample_row.get("language", "en"),
    wl=wordlists
)
print(f"Uji ekstraksi pada 1 teks (source={_sample_row.get('source','?')}, "
      f"lang={_sample_row.get('language','?')}):")
for k, v in _sample_feat.items():
    marker = " <-- EN-only, 0 untuk ID" if k == "contraction_count" else ""
    print(f"  {k:<30s} = {v:>10.5f}{marker}")
print(f"\nTotal fitur dihasilkan: {len(_sample_feat)}")


Uji ekstraksi pada 1 teks (source=ai, lang=en):
  sentence_count                 =    8.00000
  avg_sentence_length            =   19.50000
  sentence_length_std            =   10.39230
  paragraph_count                =    3.00000
  paragraph_length_std           =    9.93311
  unique_word_ratio              =    0.56410
  em_dash_density                =    0.00000
  colon_density                  =    0.00000
  semicolon_density              =    0.00000
  exclamation_density            =    0.00000
  question_density               =    0.00000
  bullet_list_count              =    0.00000
  bold_pattern_count             =    0.00000
  header_pattern_count           =    0.00000
  avg_word_length                =    4.50000
  comma_density                  =   11.36364
  parenthesis_density            =    0.00000
  contraction_count              =    0.00000 <-- EN-only, 0 untuk ID
  number_density                 =    0.00000
  capital_ratio                  =    0.07153
  ai_voc

## 5. Ekstraksi Fitur — Chunked Processing

Dataset diproses dalam potongan (*chunk*) untuk ketahanan terhadap crash.
Setiap chunk disimpan sebagai file `.parquet` terpisah di folder `feat_chunks/`.

**Resume otomatis:** Chunk yang sudah diproses pada sesi sebelumnya akan otomatis dilewati.
Klik **Mulai / Lanjutkan Ekstraksi** untuk memulai atau melanjutkan dari titik terakhir.

Klik **Hapus Semua Chunk** hanya jika ingin memulai ulang dari awal.


In [24]:
feature_df   = None
CHUNK_SIZE   = 10000

os.makedirs(CHUNKS_DIR, exist_ok=True)

def _get_done_chunks(n_chunks):
    """Kembalikan list index chunk yang sudah diproses (filenya ada)."""
    return [i for i in range(n_chunks)
            if os.path.exists(os.path.join(CHUNKS_DIR, f"chunk_{i:04d}.parquet"))]

_total_rows   = len(df_data)
_n_chunks_est = (_total_rows + CHUNK_SIZE - 1) // CHUNK_SIZE
_done_init    = _get_done_chunks(_n_chunks_est)

# Widgets
chunk_size_slider = widgets.IntSlider(
    value=10000, min=1000, max=50000, step=1000,
    description="Chunk Size:", style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)
btn_extract = widgets.Button(
    description="Mulai / Lanjutkan Ekstraksi",
    button_style="primary", icon="play",
    layout=widgets.Layout(width="270px", height="36px")
)
btn_clear_chunks = widgets.Button(
    description="Hapus Semua Chunk & Mulai Ulang",
    button_style="danger", icon="trash",
    layout=widgets.Layout(width="280px", height="36px")
)
progress_bar = widgets.HTML(value="")
out_extract  = widgets.Output()

def _render_progress(done, total):
    pct = done / max(total, 1) * 100
    progress_bar.value = (
        f"<div style='font-family:monospace;margin:6px 0'>"
        f"<b>Progress: {done}/{total} chunk selesai ({pct:.1f}%)</b><br>"
        f"<div style='background:#1a1a2e;border-radius:4px;height:16px;width:520px;margin-top:4px'>"
        f"<div style='background:#4a90e2;height:16px;width:{min(pct,100):.1f}%;"
        f"border-radius:4px;transition:width 0.2s'></div></div></div>"
    )

_render_progress(len(_done_init), _n_chunks_est)

print(f"Total baris      : {_total_rows:,}")
print(f"Estimasi chunk   : {_n_chunks_est} (chunk size {CHUNK_SIZE:,})")
print(f"Sudah diproses   : {len(_done_init)} chunk")
print(f"Sisa diproses    : {_n_chunks_est - len(_done_init)} chunk")
if _done_init:
    print(f"Klik 'Mulai / Lanjutkan' untuk melanjutkan dari chunk {max(_done_init)+1}.")


def _run_extraction(b):
    global CHUNK_SIZE
    CHUNK_SIZE  = chunk_size_slider.value
    n_chunks    = (len(df_data) + CHUNK_SIZE - 1) // CHUNK_SIZE
    active_feats = [f for f in ALL_FEATURES if _checkboxes[f].value]

    with out_extract:
        clear_output()
        if not active_feats:
            print("Tidak ada fitur yang dipilih. Aktifkan minimal 1 fitur di Bagian 3.")
            return

        print(f"Fitur aktif : {len(active_feats)}/{len(ALL_FEATURES)}")
        print(f"Chunk size  : {CHUNK_SIZE:,} baris per chunk")
        print(f"Total chunk : {n_chunks}")
        print("-" * 60)
        t_start = time.time()

        for i in range(n_chunks):
            chunk_file = os.path.join(CHUNKS_DIR, f"chunk_{i:04d}.parquet")

            if os.path.exists(chunk_file):
                # Sudah diproses sesi sebelumnya lewati
                done_count = len(_get_done_chunks(n_chunks))
                _render_progress(done_count, n_chunks)
                continue

            row_start = i * CHUNK_SIZE
            row_end   = min(row_start + CHUNK_SIZE, len(df_data))
            chunk     = df_data.iloc[row_start:row_end]

            rows = []
            for _, row in chunk.iterrows():
                feat_row = extract_features(
                    row.get("text_clean", ""),
                    lang=row.get("language", "en"),
                    wl=wordlists
                )
                feat_row["_id"]     = row.get("id", "")
                feat_row["_source"] = row.get("source", "")
                rows.append(feat_row)

            chunk_df = pd.DataFrame(rows)
            chunk_df.to_parquet(chunk_file, index=False, engine="fastparquet")

            done_count = len(_get_done_chunks(n_chunks))
            _render_progress(done_count, n_chunks)

            elapsed = time.time() - t_start
            speed   = row_end / elapsed if elapsed > 0 else 0
            eta_min = (len(df_data) - row_end) / speed / 60 if speed > 0 else 0
            print(f"  Chunk {i+1:4d}/{n_chunks} | baris {row_start:>7,}–{row_end:>7,} "
                  f"| {speed:>7,.0f} baris/det | ETA {eta_min:.1f} menit")

            gc.collect()

        total_min = (time.time() - t_start) / 60
        done_final = len(_get_done_chunks(n_chunks))
        print(f"\nEkstraksi selesai: {done_final}/{n_chunks} chunk dalam {total_min:.2f} menit.")
        if done_final == n_chunks:
            print("Semua chunk selesai. Lanjutkan ke Bagian 6 untuk menggabungkan dan menyimpan.")


def _clear_chunks(b):
    n_chunks = (_total_rows + chunk_size_slider.value - 1) // chunk_size_slider.value
    with out_extract:
        clear_output()
        removed = 0
        for i in range(n_chunks):
            fp = os.path.join(CHUNKS_DIR, f"chunk_{i:04d}.parquet")
            if os.path.exists(fp):
                os.remove(fp)
                removed += 1
        _render_progress(0, n_chunks)
        print(f"{removed} chunk dihapus. Siap memulai dari awal.")


btn_extract.on_click(_run_extraction)
btn_clear_chunks.on_click(_clear_chunks)

display(
    chunk_size_slider,
    widgets.HBox([btn_extract, btn_clear_chunks]),
    progress_bar,
    out_extract
)


Total baris      : 100,000
Estimasi chunk   : 10 (chunk size 10,000)
Sudah diproses   : 10 chunk
Sisa diproses    : 0 chunk
Klik 'Mulai / Lanjutkan' untuk melanjutkan dari chunk 10.


IntSlider(value=10000, description='Chunk Size:', layout=Layout(width='450px'), max=50000, min=1000, step=1000…

HTML(value="<div style='font-family:monospace;margin:6px 0'><b>Progress: 10/10 chunk selesai (100.0%)</b><br><…

Output()

## 6. Gabungkan Chunk & Simpan Checkpoint

Setelah semua chunk selesai diproses, gabungkan menjadi satu DataFrame dan simpan
ke `features_structural.parquet`. File ini menjadi input untuk Notebook 4 (Training).

Sel ini juga menyimpan `feature_config.json` secara otomatis dengan fitur aktif yang dipilih.


In [ ]:
btn_merge = widgets.Button(
    description="Gabungkan Chunk & Simpan Checkpoint",
    button_style="success", icon="download",
    layout=widgets.Layout(width="310px", height="36px")
)
out_merge = widgets.Output()

def _merge_and_save(b):
    global feature_df
    with out_merge:
        clear_output()
        n_chunks = (len(df_data) + CHUNK_SIZE - 1) // CHUNK_SIZE
        done = _get_done_chunks(n_chunks)

        if len(done) < n_chunks:
            missing = n_chunks - len(done)
            print(f"Belum semua chunk selesai: {len(done)}/{n_chunks} tersedia ({missing} chunk belum ada).")
            print("Jalankan ekstraksi di Bagian 5 terlebih dahulu.")
            return

        print(f"Menggabungkan {len(done)} chunk...")
        t0 = time.time()
        all_chunks = []
        for i in sorted(done):
            fp = os.path.join(CHUNKS_DIR, f"chunk_{i:04d}.parquet")
            all_chunks.append(pd.read_parquet(fp))

        feature_df = pd.concat(all_chunks, ignore_index=True)
        elapsed = time.time() - t0
        print(f"Gabungan selesai dalam {elapsed:.2f} detik. Shape: {feature_df.shape}")

        # Rename kolom identifier
        feature_df = feature_df.rename(columns={"_id": "id", "_source": "source"})

        # Susun kolom: id, source, lalu fitur aktif dalam urutan kanonik
        active_feats  = [f for f in ALL_FEATURES if _checkboxes[f].value]
        cols_in_df    = [f for f in active_feats if f in feature_df.columns]
        meta_cols     = [c for c in ["id", "source"] if c in feature_df.columns]
        final_cols    = meta_cols + cols_in_df
        feature_df    = feature_df[final_cols].copy()

        # Pastikan semua fitur bertipe float64
        for col in cols_in_df:
            feature_df[col] = feature_df[col].astype("float64")

        print(f"Kolom akhir: {meta_cols} + {len(cols_in_df)} fitur")
        print(f"Distribusi source:")
        display(feature_df["source"].value_counts().rename("Jumlah").to_frame())

        # Simpan ke parquet
        feature_df.to_parquet(CHECKPOINT_OUTPUT, index=False, engine="fastparquet")
        file_mb = os.path.getsize(CHECKPOINT_OUTPUT) / 1024 ** 2
        print(f"\nDisimpan ke: {CHECKPOINT_OUTPUT} ({file_mb:.2f} MB)")

        # Simpan feature_config.json sekalian
        config = {
            "active_features": active_feats,
            "feature_count": len(active_feats),
            "all_features_canonical": ALL_FEATURES,
            "saved_at": datetime.now().isoformat(),
            "notes": "Generated by 02_feature_engineering.ipynb. Digunakan oleh Notebook 4."
        }
        with open(FEATURE_CONFIG_FILE, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2, ensure_ascii=False)
        print(f"Disimpan ke: {FEATURE_CONFIG_FILE} ({len(active_feats)} fitur aktif)")
        print("\nSiap untuk Notebook 3 (Embedding Extraction) dan Notebook 4 (Training).")

        display(feature_df[cols_in_df].describe().round(4))

btn_merge.on_click(_merge_and_save)
display(btn_merge, out_merge)


Button(button_style='success', description='Gabungkan Chunk & Simpan Checkpoint', icon='download', layout=Layo…

Output()

## 7. Visualisasi & Analisis Fitur

Analisis statistik setiap fitur dipecah berdasarkan `source` (human vs AI).
Jalankan bagian ini setelah `feature_df` berhasil dibuat di Bagian 6.

Tersedia tiga tampilan:
1. **Tabel Statistik** — Mean, Std, dan tingkat separasi human vs AI per fitur.
2. **Heatmap Korelasi** — Korelasi antar fitur (membantu deteksi redundansi).
3. **Distribusi Fitur** — Histogram human vs AI untuk fitur pilihan.


In [26]:
btn_stats = widgets.Button(description="Tampilkan Tabel Statistik", button_style="info", icon="table")
out_stats = widgets.Output()

def _show_stats(b):
    with out_stats:
        clear_output()
        if feature_df is None:
            print("feature_df belum tersedia. Jalankan Bagian 6 terlebih dahulu.")
            return

        active = [f for f in ALL_FEATURES if f in feature_df.columns]
        human_df = feature_df[feature_df["source"] == "human"][active]
        ai_df    = feature_df[feature_df["source"] == "ai"][active]

        stats = pd.DataFrame({
            "Mean (Human)": human_df.mean(),
            "Mean (AI)":    ai_df.mean(),
            "Std (Human)":  human_df.std(),
            "Std (AI)":     ai_df.std(),
        })
        stats["Delta (AI - Human)"] = stats["Mean (AI)"] - stats["Mean (Human)"]
        # Separasi relatif: |delta| / max(|mean_H|, |mean_AI|)
        denom = stats[["Mean (Human)", "Mean (AI)"]].abs().max(axis=1) + 1e-9
        stats["Separasi %"] = (stats["Delta (AI - Human)"].abs() / denom * 100)
        stats = stats.sort_values("Separasi %", ascending=False)

        print(f"Statistik: Human ({len(human_df):,} baris) vs AI ({len(ai_df):,} baris)")
        print(f"Diurutkan berdasarkan tingkat separasi (semakin tinggi = fitur semakin diskriminatif).")
        display(
            stats.round(5).style
            .background_gradient(subset=["Separasi %"], cmap="YlOrRd")
            .format("{:.5f}", subset=["Mean (Human)", "Mean (AI)", "Std (Human)", "Std (AI)", "Delta (AI - Human)"])
            .format("{:.2f}%", subset=["Separasi %"])
        )

btn_stats.on_click(_show_stats)
display(btn_stats, out_stats)


Button(button_style='info', description='Tampilkan Tabel Statistik', icon='table', style=ButtonStyle())

Output()

In [22]:
btn_heatmap = widgets.Button(description="Tampilkan Heatmap Korelasi", button_style="info", icon="th")
out_heatmap = widgets.Output()

def _show_heatmap(b):
    with out_heatmap:
        clear_output()
        if feature_df is None:
            print("feature_df belum tersedia.")
            return

        active = [f for f in ALL_FEATURES if f in feature_df.columns]
        corr   = feature_df[active].corr()

        fig, ax = plt.subplots(figsize=(15, 12))
        sns.heatmap(
            corr, annot=False, cmap="coolwarm", center=0,
            linewidths=0.3, linecolor="#2a2a4a",
            ax=ax, square=True, vmin=-1, vmax=1
        )
        ax.set_title("Heatmap Korelasi antar Fitur (29 x 29)", fontsize=13, pad=12)
        plt.xticks(rotation=45, ha="right", fontsize=8)
        plt.yticks(rotation=0, fontsize=8)
        plt.tight_layout()
        plt.show()

        # Tampilkan pasangan korelasi tinggi (>0.7) sebagai peringatan redundansi
        high_corr = []
        for i in range(len(active)):
            for j in range(i + 1, len(active)):
                c = corr.iloc[i, j]
                if abs(c) >= 0.7:
                    high_corr.append((active[i], active[j], round(c, 3)))
        if high_corr:
            print(f"\nPasangan fitur dengan korelasi tinggi (|r| >= 0.7) pertimbangkan ablasi:")
            for a, b_, c in sorted(high_corr, key=lambda x: -abs(x[2])):
                print(f"  {a} <-> {b_}: r = {c}")
        else:
            print("\nTidak ada pasangan fitur dengan korelasi tinggi (|r| >= 0.7).")

btn_heatmap.on_click(_show_heatmap)
display(btn_heatmap, out_heatmap)


Button(button_style='info', description='Tampilkan Heatmap Korelasi', icon='th', style=ButtonStyle())

Output()

In [23]:
feat_selector = widgets.SelectMultiple(
    options=ALL_FEATURES,
    value=ALL_FEATURES[:6],
    description="Pilih Fitur:",
    rows=12,
    layout=widgets.Layout(width="360px")
)
btn_distplot = widgets.Button(description="Tampilkan Distribusi", button_style="info", icon="bar-chart")
out_dist     = widgets.Output()

def _show_distributions(b):
    with out_dist:
        clear_output()
        if feature_df is None:
            print("feature_df belum tersedia.")
            return

        selected = list(feat_selector.value)
        if not selected:
            print("Pilih minimal 1 fitur.")
            return

        n     = len(selected)
        ncols = 2
        nrows = (n + ncols - 1) // ncols

        fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
        axes = np.array(axes).flatten()

        human_data = feature_df[feature_df["source"] == "human"]
        ai_data    = feature_df[feature_df["source"] == "ai"]

        for idx, feat in enumerate(selected):
            ax = axes[idx]
            h_vals = human_data[feat].dropna()
            a_vals = ai_data[feat].dropna()

            # Clip pada persentil ke-99 agar outlier tidak mendominasi tampilan
            q99 = max(h_vals.quantile(0.99), a_vals.quantile(0.99), 1e-9)

            ax.hist(h_vals.clip(upper=q99), bins=50, alpha=0.55,
                    color="#4a90e2", label=f"Human (n={len(h_vals):,})", density=True)
            ax.hist(a_vals.clip(upper=q99), bins=50, alpha=0.55,
                    color="#e27a4a", label=f"AI (n={len(a_vals):,})",    density=True)

            ax.axvline(h_vals.median(), color="#4a90e2", linestyle="--", linewidth=1.8, alpha=0.9)
            ax.axvline(a_vals.median(), color="#e27a4a", linestyle="--", linewidth=1.8, alpha=0.9)

            ax.set_title(feat, fontsize=10, fontweight="bold")
            ax.set_xlabel("Nilai fitur")
            ax.set_ylabel("Densitas")
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.2)

        for j in range(len(selected), len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(
            "Distribusi Fitur: Human (biru) vs AI (oranye) | Garis putus-putus = median",
            fontsize=11, y=1.02
        )
        plt.tight_layout()
        plt.show()

btn_distplot.on_click(_show_distributions)
display(
    widgets.HBox([
        feat_selector,
        widgets.VBox([
            widgets.HTML("<b>Pilih fitur (Ctrl+click untuk multi-pilih):</b>"),
            btn_distplot
        ])
    ]),
    out_dist
)


Output()